# 10.3 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/10-deployment-mlops/03-streamlit/exercises.ipynb)

In [ ]:
# --- course files: found locally, downloaded on Colab -------------------------
import sys, urllib.request
from pathlib import Path
REPO_RAW = "https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/"

def repo_file(rel):
    """Return a local path to a file in this repo, downloading it if needed."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists():
            return parent / rel
    dest = Path("course_files") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
repo_file("data/download.py")                                  # churn_model finds the data through it
sys.path.insert(0, str(repo_file("lib/churn_model.py").parent))
import churn_model as cm

bundle = cm.load_bundle()          # trains and saves models/churn_bundle.joblib the first time
ROOT = cm.repo_root()
print(f"Model trained {bundle['trained']}, test AUC {bundle['metrics']['test_auc']}, "
      f"threshold {bundle['threshold']}")

In [ ]:
scaler, logreg = bundle["pipeline"][0], bundle["pipeline"][-1]

def contributions(values):
    """Each feature's push on the log-odds: weight × standardised value."""
    z = (np.asarray(values, dtype=float) - scaler.mean_) / scaler.scale_
    return pd.Series(logreg.coef_[0] * z, index=bundle["features"])

def probability(values):
    return 1 / (1 + np.exp(-(logreg.intercept_[0] + contributions(values).sum())))

## Exercise 1: by hand

Compute the contributions and probability for 1 order, £80 spend, 3 products, 273 days tenure, 273 days since the last order. Which feature dominates? Check with `contributions()` and `probability()`.

In [ ]:
# your code here

## Exercise 2: a CSV tab

Copy `app.py` to `app_batch.py` in this folder and add `tab1, tab2 = st.tabs(["One customer", "Upload a CSV"])`. In the second tab, use `st.file_uploader` and show a table of churn probabilities and voucher decisions. Run it with `python -m streamlit run app_batch.py`.

In [ ]:
# your code here (or work in the .py file and run it from a terminal)

## Exercise 3: which threshold?

Find, among the 830 test customers, those where the break-even rule (5/60) and the bundle threshold (0.07) disagree. How many are there, and what do they have in common?

In [ ]:
# your code here